In [11]:
# Imports
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load CSVs
train_df = pd.read_csv("train_df_processed.csv")
val_df = pd.read_csv("val_df_processed.csv")

# Class labels mapping
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
class_to_idx = {c: i for i, c in enumerate(class_columns)}

# Data augmentation transforms for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Validation transforms - no augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset class
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label_idx']
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Create datasets and dataloaders
train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)


In [1]:
# Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

from tqdm import tqdm
import time

device = torch.device("cpu")

In [3]:
train_df = pd.read_csv("train_split.csv")
val_df = pd.read_csv("val_split.csv")
print(train_df.columns)


train_df['image_path'] = train_df['image'].apply(lambda x: f"resizedTrainingData/{x}.jpg")
val_df['image_path'] = val_df['image'].apply(lambda x: f"resizedValidationData/{x}.jpg")

# Label mapping
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
idx_to_class = {i: col for i, col in enumerate(class_columns)}

train_df.to_csv("train_df_processed.csv", index=False)
val_df.to_csv("val_df_processed.csv", index=False)


Index(['image', 'MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC', 'label_idx',
       'label_name', 'image_path'],
      dtype='object')


In [4]:
# Image transforms (same as before)
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']  # ✅ This should match your column
        image = Image.open(img_path).convert("RGB")
        label = self.df.iloc[idx]['label_idx']
        if self.transform:
            image = self.transform(image)
        return image, label


# Dataloaders
train_dataset = SkinLesionDataset(train_df, transform=image_transforms)
val_dataset = SkinLesionDataset(val_df, transform=image_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [5]:
# Load pre-trained ResNet18
model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 7)  # 7 classes

# Send to CPU
model = model.to(device)


/Users/anushekhan/skinLesionIdentifierProject/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/anushekhan/skinLesionIdentifierProject/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)  # Only the final layer will be trained


In [7]:
from tqdm import tqdm
import time

def train_model(model, criterion, optimizer, train_loader, val_loader, scheduler=None, epochs=10):
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_loss, correct, total = 0, 0, 0

        print(f"\nEpoch {epoch+1}/{epochs}")
        loop = tqdm(train_loader, desc="Training", leave=False)

        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            loop.set_postfix(loss=loss.item())

        train_acc = correct / total
        avg_loss = total_loss / len(train_loader)

        # Validation phase
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        if scheduler:
            scheduler.step()

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Time: {epoch_time:.2f}s")


In [8]:
def evaluate_model(model, val_loader, class_names):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    print(classification_report(y_true, y_pred, target_names=class_names))
    return y_true, y_pred


In [9]:
# Save split DataFrames for later use
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)


In [10]:
from PIL import Image
import os

orig_val_folder = "validationData"
resized_val_folder = "resizedValidationData"
os.makedirs(resized_val_folder, exist_ok=True)

# List all images in original validation folder
all_val_images = [f for f in os.listdir(orig_val_folder) if f.endswith(".jpg")]

print(f"Resizing {len(all_val_images)} images...")

for fname in all_val_images:
    orig_path = os.path.join(orig_val_folder, fname)
    new_path = os.path.join(resized_val_folder, fname)
    if not os.path.exists(new_path):  # skip if already resized
        img = Image.open(orig_path).convert("RGB")
        img = img.resize((224, 224))
        img.save(new_path)

print("Resizing complete!")


Resizing 2196 images...
Resizing complete!
